# Enterprise Production Capabilities Test Suite

Tests PDF Upload, Automatic Document Ingestion, Streaming SSE Responses, Chart-Ready SQL Results, Auth + RBAC, Observability, and Full System Regression.

In [ ]:
import sys
from pathlib import Path

# Resolve project root
cwd = Path.cwd().resolve()
if cwd.name == "tests":
    project_root = cwd.parent.parent
elif cwd.name == "backend":
    project_root = cwd.parent
else:
    project_root = cwd

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from fastapi.testclient import TestClient
from backend.main import app
from backend.app.tools.sql_tool import execute_sql_query
from backend.app.rag.retriever import retrieve

client = TestClient(app)
print("FastAPI TestClient initialized with all Enterprise Production Features!")

In [ ]:
# SECTION 1: AUTHENTICATION + RBAC TESTS
# 1. Register normal user
r_reg_user = client.post("/api/auth/register", json={
    "username": "testuser_001",
    "email": "testuser_001@nexa.com",
    "password": "SecurePass123!",
    "role": "user"
})
user_token = r_reg_user.json()["access_token"]
print("Registered User Token:", user_token[:30] + "...")

# 2. Register admin user
r_reg_admin = client.post("/api/auth/register", json={
    "username": "adminuser_001",
    "email": "adminuser_001@nexa.com",
    "password": "AdminPass123!",
    "role": "admin"
})
admin_token = r_reg_admin.json()["access_token"]
print("Registered Admin Token:", admin_token[:30] + "...")

# 3. Protected endpoint without JWT -> 401
r_unauth = client.get("/api/auth/me")
assert r_unauth.status_code == 401

# 4. Authenticated /me endpoint -> 200
r_me = client.get("/api/auth/me", headers={"Authorization": f"Bearer {user_token}"})
assert r_me.status_code == 200
assert r_me.json()["username"] == "testuser_001"

# 5. Admin-only endpoint with user -> 403 Forbidden
r_user_admin_ep = client.get("/api/documents", headers={"Authorization": f"Bearer {user_token}"})
assert r_user_admin_ep.status_code == 403

# 6. Admin-only endpoint with admin -> 200 OK
r_admin_ep = client.get("/api/documents", headers={"Authorization": f"Bearer {admin_token}"})
assert r_admin_ep.status_code == 200
print("Auth & RBAC Tests: ALL 6 TESTS PASSED!")

In [ ]:
# SECTION 2: PDF UPLOAD & AUTO INGESTION TESTS
import io

# Minimal valid PDF bytes for testing
pdf_bytes = b"%PDF-1.4\n1 0 obj\n<< /Type /Catalog /Pages 2 0 R >>\nendobj\n2 0 obj\n<< /Type /Pages /Kids [3 0 R] /Count 1 >>\nendobj\n3 0 obj\n<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Contents 4 0 R >>\nendobj\n4 0 obj\n<< /Length 55 >>\nstream\nBT /F1 12 Tf 100 700 Td (NexaTech Security Policy 2026) Tj ET\nendstream\nendobj\nxref\n0 5\n0000000000 65535 f \n0000000009 00000 n \n0000000058 00000 n \n0000000115 00000 n \n0000000216 00000 n \ntrailer\n<< /Size 5 /Root 1 0 R >>\nstartxref\n320\n%%EOF"

# 1. Upload valid PDF
files = {"file": ("security_policy_2026.pdf", io.BytesIO(pdf_bytes), "application/pdf")}
r_up = client.post("/api/documents/upload", files=files, headers={"Authorization": f"Bearer {user_token}"})
print("PDF Upload Response:", r_up.json())
assert r_up.status_code == 200
assert r_up.json()["status"] == "ingested"

# 2. Invalid file type upload
bad_files = {"file": ("malicious.txt", io.BytesIO(b"plain text"), "text/plain")}
r_bad_up = client.post("/api/documents/upload", files=bad_files, headers={"Authorization": f"Bearer {user_token}"})
assert r_bad_up.status_code == 400

In [ ]:
# SECTION 3: STREAMING SSE RESPONSES
r_stream = client.post("/api/chat/stream", json={"message": "What is Python?"})
assert r_stream.status_code == 200
assert "text/event-stream" in r_stream.headers["content-type"]
stream_content = r_stream.text
print("SSE Stream Chunk Preview:", stream_content[:200])
assert "route_selected" in stream_content
assert "token" in stream_content

In [ ]:
# SECTION 4: CHART-READY SQL RESULTS
# 1. Scalar query -> no chart
res_scalar = execute_sql_query("SELECT COUNT(*) FROM sales;")
print("Scalar SQL Result:", res_scalar)
assert res_scalar["chart"] is None

# 2. Tabular grouped query -> chart metadata generated
res_grouped = execute_sql_query("SELECT region, SUM(amount) AS total_revenue FROM sales GROUP BY region;")
print("Grouped SQL Result with Chart:", res_grouped["chart"])
assert res_grouped["chart"] is not None
assert "chart_type" in res_grouped["chart"]
assert "data" in res_grouped["chart"]

In [ ]:
# SECTION 5: OBSERVABILITY & CORRELATION ID
r_obs = client.post("/api/chat", json={"message": "Hello"}, headers={"X-Request-ID": "test-correlation-12345"})
assert r_obs.headers.get("X-Request-ID") == "test-correlation-12345"
print("Observability X-Request-ID Header Verified:", r_obs.headers.get("X-Request-ID"))